In [ ]:
import subprocess
import sys
from pathlib import Path
import shutil

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import torch
from ultralytics import YOLO
import pandas as pd

# Project root is one level up from models/
PROJECT_ROOT = Path("__file__").resolve().parent.parent
# When running the notebook directly from models/ dir:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "models" else Path.cwd()

CHECKPOINTS = PROJECT_ROOT / "models" / "checkpoints"
YAMLS = PROJECT_ROOT / "data" / "processed" / "yamls"

CHECKPOINTS.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Checkpoints: {CHECKPOINTS}")
print(f"YAMLs: {YAMLS}")

In [ ]:
def get_device():
    if torch.backends.mps.is_available():
        return "mps"
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"

DEVICE = get_device()

In [ ]:
required_yamls = [
    YAMLS / "stage1_quadrant.yaml",
    YAMLS / "stage2_enumeration.yaml",
    YAMLS / "stage3_disease.yaml",
]
for y in required_yamls:
    status = "✓" if y.exists() else "✗ MISSING"
    print(f"  {status}  {y.name}")

if not all(y.exists() for y in required_yamls):
    raise RuntimeError("Run `python preprocessing.py` from the project root first.")

In [ ]:
EPOCHS_S1 = 100 # Stage 1 — quadrant (large data, can afford more epochs)
EPOCHS_S2 = 100 # Stage 2 — enumeration
EPOCHS_S3 = 100 # Stage 3 — diagnosis
EPOCHS_BL = 100 # Baseline — must match Stage 3 for fair comparison

BATCH = 8 # 8 works on MPS 16GB; increase to 16 on CUDA
IMGSZ = 640
PATIENCE = 20 # early stopping (25 for stage3/baseline — small dataset)

In [ ]:
# Generic training wrapper.
# Returns path to best.pt
def run_stage(name, model_path, yaml_path, run_name, epochs, lr0,
              patience=20, copy_paste=0.0, out_name=None):
    model = YOLO(str(model_path))
    results = model.train(
        data=str(yaml_path),
        epochs=epochs,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        project=str(CHECKPOINTS / "runs"),
        name=run_name,
        exist_ok=True,
        optimizer="AdamW",
        lr0=lr0,
        lrf=0.01,
        warmup_epochs=3,
        weight_decay=0.0005,
        augment=True,
        hsv_s=0.4, hsv_v=0.4,
        flipud=0.1, fliplr=0.5,
        mosaic=1.0,
        copy_paste=copy_paste,
        patience=patience,
        save=True,
        plots=True,
    )
    best_src = Path(results.save_dir) / "weights" / "best.pt"
    best_dst = CHECKPOINTS / (out_name or f"{run_name}_best.pt")
    if best_src.exists():
        shutil.copy(best_src, best_dst)
        print(f"\n✓ {name} checkpoint → {best_dst}")
    return best_dst, results

In [ ]:
s1_ckpt, s1_results = run_stage(
    name = "Stage 1 — Quadrant Detection",
    model_path = "yolov8m-seg.pt", # ImageNet pretrained
    yaml_path = YAMLS / "stage1_quadrant.yaml",
    run_name = "stage1",
    epochs = EPOCHS_S1,
    lr0 = 0.001,
    out_name = "stage1_best.pt",
)

In [ ]:
s2_ckpt, s2_results = run_stage(
    name = "Stage 2 — Tooth Enumeration",
    model_path = CHECKPOINTS / "stage1_best.pt",
    yaml_path = YAMLS / "stage2_enumeration.yaml",
    run_name = "stage2",
    epochs = EPOCHS_S2,
    lr0 = 0.0005,
    out_name = "stage2_best.pt",
)

In [ ]:
s3_ckpt, s3_results = run_stage(
    name = "Stage 3 — Diagnosis Detection (Curriculum)",
    model_path = CHECKPOINTS / "stage2_best.pt",
    yaml_path = YAMLS / "stage3_disease.yaml",
    run_name = "stage3_curriculum",
    epochs = EPOCHS_S3,
    lr0 = 0.0003,
    patience = 25,
    copy_paste = 0.1,
    out_name = "stage3_best.pt",
)

In [ ]:
bl_ckpt, bl_results = run_stage(
    name = "Baseline — Single-Stage",
    model_path = "yolov8m-seg.pt", # No curriculum warm-start
    yaml_path = YAMLS / "stage3_disease.yaml",
    run_name = "baseline",
    epochs = EPOCHS_BL,
    lr0 = 0.001,
    patience   = 25,
    copy_paste = 0.1,
    out_name   = "baseline_best.pt",
)

In [ ]:
checkpoints = [
    ("Stage 1", "stage1_best.pt"),
    ("Stage 2", "stage2_best.pt"),
    ("Stage 3", "stage3_best.pt"),
    ("Baseline", "baseline_best.pt"),
]

rows = []
for label, fname in checkpoints:
    p = CHECKPOINTS / fname
    size_mb = p.stat().st_size / 1e6 if p.exists() else None
    rows.append({"Model": label, "File": fname,
                 "Exists": "✓" if p.exists() else "✗",
                 "Size (MB)": f"{size_mb:.1f}" if size_mb else "—"})

df = pd.DataFrame(rows)
print(df.to_string(index=False))

In [ ]:
runs = {
    "Stage 3 (Curriculum)": CHECKPOINTS / "runs" / "stage3_curriculum",
    "Baseline": CHECKPOINTS / "runs" / "baseline",
}

fig, axes = plt.subplots(1, len(runs), figsize=(14, 6))
for ax, (label, run_dir) in zip(axes, runs.items()):
    results_png = run_dir / "results.png"
    if results_png.exists():
        img = mpimg.imread(results_png)
        ax.imshow(img)
        ax.set_title(label, fontsize=12)
    else:
        ax.text(0.5, 0.5, f"No results.png\n{run_dir}",
                ha="center", va="center", transform=ax.transAxes)
    ax.axis("off")

plt.suptitle("Training Curves", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Next steps

Open **`02_evaluation.ipynb`** to:
- Compute mAP@0.5 and mAP@0.5:0.95 on the test set
- Compare curriculum vs. baseline (RQ1)
- Analyse per-class performance (RQ3)
- Visualise predictions on sample X-rays